# LIBERO-Mem-derived V1 · SmolVLA memory adapter

Attach `mem_adapter_v1_code.zip` and the data ZIP with its JSON sidecar as private Kaggle inputs. Enable Internet for the pinned dependencies and assets. Select GPU T4 ×2 for the default recipe.

With `prepare_derived_on_kaggle`, attach `libero_mem_derived_v1_sources.zip`. The notebook verifies the sources, generates RGB memory for T1/T3 and builds the shards. Use `attached_prepared` with `mem_adapter_v1_data.zip` or the extracted dataset when the prepared shards are available.

The pinned split contains 500 LIBERO-10 base demonstrations and 188 LIBERO-Mem-derived T1/T3 demonstrations. The experiment is **NON-COMPARABLE_OFFICIAL_PAPER**.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

DATA_MODE = 'prepare_derived_on_kaggle'  # or 'attached_prepared'
GPUS = 2
if DATA_MODE not in {'prepare_derived_on_kaggle', 'attached_prepared'} or GPUS not in {1, 2}:
    raise ValueError('Set an explicit data mode and GPU count')
INPUT = Path('/kaggle/input')
SCRATCH = Path('/kaggle/temp/libero-mem-derived-v1')
WORK = Path('/kaggle/working/libero-mem-derived-v1')
REPO = SCRATCH / 'code'
TRAIN_PY = SCRATCH / 'training-env/bin/python'
ASSETS = SCRATCH / 'assets'
PERCEPTION_PY = SCRATCH / 'perception-env/bin/python'
PREP = WORK / 'preparation'
EXPECTED_LOCK = '3d9be02b8861481f0b4379697649198c01c36aec5915f6f50b9e53c23e0da2cf'
for directory in (SCRATCH, WORK):
    directory.mkdir(parents=True, exist_ok=True)

def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def run(python, *args):
    subprocess.run([str(python), *map(str, args)], cwd=REPO, check=True)

def extract_once(archive, target, sha256):
    receipt = target / '.archive.sha256'
    if target.exists():
        if not receipt.is_file() or receipt.read_text().strip() != sha256:
            raise ValueError(f'Scratch artifact differs from attached ZIP: {target}')
        return
    with zipfile.ZipFile(archive) as bundle:
        members = bundle.infolist()
        if sum(member.file_size for member in members) > shutil.disk_usage(target.parent).free:
            raise RuntimeError(f'Insufficient scratch disk to extract {archive.name}')
        with tempfile.TemporaryDirectory(dir=target.parent) as temp:
            stage = Path(temp) / target.name
            stage.mkdir()
            paths = [(stage / member.filename).resolve() for member in members]
            if len(set(paths)) != len(paths) or any(not p.is_relative_to(stage) for p in paths):
                raise ValueError(f'Invalid ZIP member paths: {archive.name}')
            bundle.extractall(stage)
            (stage / receipt.name).write_text(sha256 + '\n')
            stage.rename(target)

code_archives = sorted(INPUT.rglob('mem_adapter_v1_code.zip'))
if len(code_archives) != 1:
    raise ValueError('Attach exactly one mem_adapter_v1_code.zip')
code_hash = digest(code_archives[0])
extract_once(code_archives[0], REPO, code_hash)
(WORK / 'code.sha256').write_text(code_hash + '\n')
print('Code archive SHA-256', code_hash)

def attach_zip(archive, target):
    sidecar = archive.with_suffix('.json')
    receipt = json.loads(sidecar.read_text(encoding='utf-8'))
    archive_hash = digest(archive)
    if (receipt.get('status') != 'complete' or receipt.get('archive') != archive.name
            or receipt.get('bytes') != archive.stat().st_size or receipt.get('sha256') != archive_hash):
        raise ValueError(f'ZIP does not match its checksum sidecar: {archive.name}')
    extract_once(archive, target, archive_hash)
    shutil.copy2(sidecar, WORK / sidecar.name)
    return receipt


## Verify the attached dataset or derived sources

Put each data ZIP and its matching JSON file in the same attached directory. For the prepared mode you can also attach the extracted dataset directory.


In [ ]:
if DATA_MODE == 'prepare_derived_on_kaggle':
    archives = sorted(INPUT.rglob('libero_mem_derived_v1_sources.zip'))
    if len(archives) != 1:
        raise ValueError('Attach exactly one derived source ZIP with its JSON sidecar')
    bundle = SCRATCH / 'sources'
    receipt = attach_zip(archives[0], bundle)
    SPLIT_LOCK = bundle / 'split.lock.json'
    MEM_ROOT = bundle / 'derived_raw'
    lock = json.loads(SPLIT_LOCK.read_text(encoding='utf-8'))
    if (receipt['split_lock_sha256'] != EXPECTED_LOCK
            or lock.get('lock_sha256') != EXPECTED_LOCK
            or lock.get('derived_manifest_sha256') != receipt['derived_manifest_sha256']
            or digest(MEM_ROOT / 'manifest.json') != lock.get('derived_manifest_file_sha256')):
        raise ValueError('Source archive does not match the pinned derived split')
    DATA = SCRATCH / 'dataset'
else:
    archives = sorted(INPUT.rglob('mem_adapter_v1_data.zip'))
    directories = []
    for path in INPUT.rglob('manifest.json'):
        try:
            manifest = json.loads(path.read_text(encoding='utf-8'))
        except (ValueError, OSError):
            continue
        if (manifest.get('artifact') == 'mixed_base_mem_sft_shards'
                and manifest.get('split_lock_sha256') == EXPECTED_LOCK):
            directories.append(path.parent)

    if len(archives) + len(directories) != 1:
        raise ValueError('Attach one derived data ZIP with its sidecar, or one extracted derived dataset')

    if archives:
        archive = archives[0]
        receipt = attach_zip(archive, SCRATCH / 'attached-data')
        DATA = SCRATCH / 'attached-data/dataset'
        if digest(DATA / 'manifest.json') != receipt['dataset_manifest_sha256']:
            raise ValueError('Extracted manifest differs from the data ZIP sidecar')
    else:
        DATA = directories[0]


## Install the pinned environments and fetch model assets

The training and perception dependencies use separate environments. The prepared mode needs only the training environment.


In [ ]:
def install(python, requirements):
    if not python.is_file():
        run(sys.executable, '-m', 'venv', python.parent.parent)
    run(python, '-m', 'pip', 'install', '--no-cache-dir', '-r', REPO / requirements)
    freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
    (WORK / (python.parent.parent.name + '-pip-freeze.txt')).write_text(freeze)

install(TRAIN_PY, 'envs/mem_adapter_training.txt')
run(TRAIN_PY, '-c', 'import torch, sys; n = int(sys.argv[1]); assert torch.cuda.device_count() == n; print([torch.cuda.get_device_name(i) for i in range(n)])', GPUS)
run(TRAIN_PY, REPO / 'scripts/memory_sft_assets.py', 'fetch', ASSETS)
if DATA_MODE == 'prepare_derived_on_kaggle':
    install(PERCEPTION_PY, 'envs/memory_perception.txt')


## Build and validate the training shards

The source mode fetches or reuses the ten pinned LIBERO-10 base files. Memory preparation writes its traces and progress under `/kaggle/working/libero-mem-derived-v1/preparation`. Save a completed notebook output version to preserve finished indices across sessions. The two tasks run on separate GPUs when two are configured.


In [ ]:
if DATA_MODE == 'prepare_derived_on_kaggle' and not DATA.exists():
    from concurrent.futures import ThreadPoolExecutor

    PREP.mkdir(parents=True, exist_ok=True)
    identity = {'code_sha256': code_hash, 'source_sha256': receipt['sha256'], 'split_lock_sha256': EXPECTED_LOCK}
    prep_receipt = PREP / 'inputs.json'
    if prep_receipt.is_file() and json.loads(prep_receipt.read_text()) != identity:
        raise ValueError('Existing preparation was created with different code or data')
    if not prep_receipt.exists() and any(PREP.iterdir()):
        raise ValueError('Existing preparation has no provenance receipt')
    prep_receipt.write_text(json.dumps(identity, indent=2) + '\n')

    BASE_ROOT = SCRATCH / 'base'
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_training_sources.py', '--base-only',
        '--base-root', BASE_ROOT, '--input-root', INPUT, '--report', WORK / 'base-sources.json')
    DETECTOR = SCRATCH / 'grounding-dino-tiny'
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_detector.py', '--output', DETECTOR)
    protocol = json.loads((REPO / 'configs/benchmarks/libero_mem_t1_t3_protocol.json').read_text())
    tasks = [(task['alias'], task['name']) for task in protocol['tasks']]

    def prepare_task(item):
        index, (alias, task) = item
        environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index % GPUS))
        subprocess.run([
            str(PERCEPTION_PY), '-u', str(REPO / 'scripts/prepare_libero_mem_sft.py'),
            '--source-kind', 'derived_raw', '--data-dir', str(MEM_ROOT),
            '--split-lock', str(SPLIT_LOCK), '--task-name', task,
            '--tokenizer-path', str(ASSETS / 'backbone'), '--model-dir', str(DETECTOR),
            '--device', 'cuda', '--output-dir', str(PREP / ('memory_' + alias)), '--resume',
        ], cwd=REPO, env=environment, check=True)

    with ThreadPoolExecutor(max_workers=GPUS) as pool:
        list(pool.map(prepare_task, enumerate(tasks)))

    memory_summary = {}
    for task in protocol['tasks']:
        alias = task['alias']
        index_manifest = json.loads((PREP / ('memory_' + alias) / 'manifest.json').read_text())
        memory_summary[alias] = {}
        for split in ('train', 'val', 'test'):
            episodes = [row for row in index_manifest['episodes'] if row['split'] == split]
            memory_summary[alias][split] = {
                'episodes': len(episodes),
                'frames': sum(row['frames'] for row in episodes),
                'memory_frames': sum(row['memory_frames'] for row in episodes),
                'episodes_with_complete_history': sum(bool(row['history_complete']) for row in episodes),
                'episodes_with_expected_return_count': sum(row['confirmed_returns'] == task['target_cycles'] for row in episodes),
            }
    (WORK / 'memory-summary.json').write_text(json.dumps(memory_summary, indent=2) + '\n')
    print(json.dumps(memory_summary, indent=2))

    with tempfile.TemporaryDirectory(dir=SCRATCH) as temporary:
        stage = Path(temporary) / 'dataset'
        run(PERCEPTION_PY, REPO / 'scripts/prepare_mixed_memory_sft.py',
            '--split-lock', SPLIT_LOCK, '--base-root', BASE_ROOT, '--mem-root', MEM_ROOT,
            '--tokenizer-path', ASSETS / 'backbone', '--output-dir', stage,
            '--memory-index', *(PREP / ('memory_' + alias) for alias, _ in tasks))
        stage.rename(DATA)

manifest = json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
lock = json.loads((DATA / 'split.lock.json').read_text(encoding='utf-8'))
if (lock.get('lock_sha256') != EXPECTED_LOCK or manifest.get('split_lock_sha256') != EXPECTED_LOCK
        or lock.get('classification') != 'CUSTOM_V2_LIBERO_MEM_DERIVED_TRAINING_DATA'
        or lock.get('mem_image_convention') != 'raw_opengl'
        or manifest.get('memory_source') != 'predicted_rgb_v3'):
    raise ValueError('The attached data does not match the pinned derived training dataset')
print('Dataset', DATA)
print('Frame counts', manifest['counts'])
run(TRAIN_PY, REPO / 'scripts/validate_mixed_memory_sft.py', '--data-dir', DATA)
run(TRAIN_PY, '-c', """
import sys
from torch.utils.data import DataLoader
from src.training.memory_dataset import MemoryDataset, collate_memory

data = MemoryDataset(sys.argv[1], 'train')
try:
    indices = [next(i for i, (episode, step) in enumerate(data.index)
                    if data.episodes[episode]['domain'] == domain and step == data.episodes[episode]['length'] - 1)
               for domain in ('base', 'mem')]
    batch = next(iter(DataLoader(data, batch_size=2, sampler=indices, collate_fn=collate_memory)))
    if (tuple(batch['action'].shape) != (2, 50, 7)
            or tuple(batch['observation.state'].shape) != (2, 8)
            or not batch['action_is_pad'][:, 1:].all()
            or batch['action_is_pad'][:, 0].any()):
        raise ValueError('The SFT loader disagrees with the pinned state/action and padding contract')
    print('Training loader preflight passed for base and derived Mem')
finally:
    data.close()
""", DATA)
shutil.copy2(DATA / 'manifest.json', WORK / 'dataset-manifest.json')
shutil.copy2(DATA / 'split.lock.json', WORK / 'dataset-split.lock.json')


## Train the SmolVLA memory adapter

The pinned recipe uses an equal mix of base and Mem samples per optimizer batch. Save the notebook output version to retain the adapter, validation metrics and provenance.


In [ ]:
from datetime import datetime, timezone

OUTPUT = WORK / ('mem-adapter-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
run(TRAIN_PY, REPO / 'scripts/train_mem_adapter.py', '--data-dir', DATA,
    '--assets-dir', ASSETS, '--output-dir', OUTPUT, '--gpus', GPUS)
print('Trained adapter', OUTPUT / 'pretrained_model')
